# AIC-2026 — MetaCLIP 2 keyframe embeddings

Notebook Colab độc lập cho GPU A100 40 GB. Pipeline clone project và source MetaCLIP, xác thực Google Drive, tải 873 ZIP từ hai folder (31.0 GB), giải nén an toàn, embed từng video, lưu shard có resume, rồi ghép ma trận cuối.

Checkpoint mặc định: `facebook/metaclip-2-worldwide-huge-quickgelu`. Vector được L2-normalize và lưu `float16`; output nằm trong Google Drive để không mất khi runtime ngắt. Chọn **Runtime → Change runtime type → A100 GPU** trước khi chạy toàn bộ.

In [ ]:
# 1) Clone project và implementation chính thức
import os
from pathlib import Path

PROJECT_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
PROJECT_BRANCH = "feat/keyframe-youtube-mapping"
PROJECT_DIR = Path("/content/AIC-2026-keyframe-embeddings")
METACLIP_REPO = Path("/content/MetaCLIP")

if not (PROJECT_DIR / ".git").exists():
    !git clone --depth 1 --branch {PROJECT_BRANCH} --single-branch {PROJECT_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} switch {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} pull --ff-only origin {PROJECT_BRANCH}

if not (METACLIP_REPO / ".git").exists():
    !git clone --depth 1 https://github.com/facebookresearch/MetaCLIP.git {METACLIP_REPO}
else:
    !git -C {METACLIP_REPO} pull --ff-only

%cd {PROJECT_DIR}

In [ ]:
# 2) Cài dependency. Không thay PyTorch CUDA do Colab cung cấp.
%pip install -q -U "transformers>=4.56.2,<5" "accelerate>=1.2" google-api-python-client google-auth-httplib2 tqdm Pillow requests

In [ ]:
# 3) Mount Drive để giữ output và cấp quyền đọc hai shared folders
from google.colab import auth, drive

drive.mount("/content/drive")
auth.authenticate_user()
print("Google Drive authentication ready")

In [ ]:
# 4) Cấu hình chạy
from pathlib import Path
import shutil
import torch

FOLDER_IDS = [
    "1ZjLlGH0Igq70wrAELVIU4kLIWSFUAN4B",  # 439 ZIP ở snapshot đã kiểm tra
    "1nxum5Qp5_iCQIqud11I8u8ACKE8OOsv5",  # 434 ZIP ở snapshot đã kiểm tra
]
DATA_DIR = Path("/content/aic_keyframes")          # SSD local: đọc ảnh nhanh
ZIP_CACHE_DIR = Path("/content/aic_zip_cache")     # mỗi ZIP bị xóa sau khi giải nén
OUTPUT_DIR = Path("/content/drive/MyDrive/AIC-2026/embeddings/metaclip2")
BATCH_SIZE = 128                                      # giảm còn 64 nếu runtime không phải A100 40 GB
NUM_WORKERS = 4
MAX_ZIPS = None                                       # đặt 2 để smoke test, None để chạy đủ 873 ZIP

assert torch.cuda.is_available(), "Hãy bật GPU runtime trong Colab"
free_gib = shutil.disk_usage("/content").free / 1024**3
print(torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")
print(f"Local disk còn {free_gib:.1f} GiB")
if MAX_ZIPS is None and free_gib < 40:
    raise RuntimeError("Cần tối thiểu khoảng 40 GiB local disk trống cho toàn bộ keyframe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 5) Tải dữ liệu + embed. Có thể chạy lại: shard video hợp lệ sẽ được bỏ qua.
import subprocess
import sys

SCRIPT_PATH = PROJECT_DIR / "scripts" / "colab_keyframe_embeddings.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {SCRIPT_PATH}. Hãy commit + push file scripts/colab_keyframe_embeddings.py "
        "lên đúng GitHub repo/branch rồi chạy lại cell clone."
    )

command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--model", "metaclip2",
    "--data-dir", str(DATA_DIR),
    "--zip-cache-dir", str(ZIP_CACHE_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
for folder_id in FOLDER_IDS:
    command += ["--folder-id", folder_id]
if MAX_ZIPS is not None:
    command += ["--max-zips", str(MAX_ZIPS), "--allow-count-mismatch"]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
# 6) Kiểm tra artifact cuối
import json
import numpy as np

manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
vectors = np.load(OUTPUT_DIR / "keyframes_visual_vectors.f16.npy", mmap_mode="r")
metadata_count = sum(1 for _ in (OUTPUT_DIR / "keyframes_metadata.jsonl").open(encoding="utf-8"))
sample = vectors[np.linspace(0, len(vectors) - 1, min(1000, len(vectors)), dtype=int)].astype(np.float32)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("matrix:", vectors.shape, vectors.dtype)
print("metadata rows:", metadata_count)
print("sample norm range:", np.linalg.norm(sample, axis=1).min(), np.linalg.norm(sample, axis=1).max())
assert vectors.shape[0] == metadata_count == manifest["keyframe_count"]
assert np.isfinite(sample).all()
assert np.allclose(np.linalg.norm(sample, axis=1), 1.0, atol=2e-3)
print("✅ MetaCLIP 2 artifacts verified")

## Output

- `shards/Lxx_Vxxx.f16.npy`: checkpoint/resume theo video.
- `keyframes_visual_vectors.f16.npy`: ma trận toàn bộ keyframe theo thứ tự metadata.
- `keyframes_metadata.jsonl`: ánh xạ chính xác row → video/keyframe/path.
- `drive_archives_manifest.json`: snapshot 873 ZIP đầu vào.
- `run_manifest.json`: model, dimension, preprocessing, count và GPU.

Dữ liệu ảnh nằm trên SSD tạm `/content`; chỉ xóa sau khi cell kiểm tra hoàn tất. Nếu đổi checkpoint/model, hãy dùng một `OUTPUT_DIR` mới để không trộn shard khác không gian vector.